# Reading the October 2022 hotspot observations

Compare the same 5.12 km grid size at eight sites. Each panel shows the closest eligible Sentinel-1 pass to 15 October. Historical water occurrence helps interpret a detection. A class boundary can reflect missing historical observations as well as landscape change.

In [1]:
from pathlib import Path
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from floodrisk.deploy.hotspot_tiles import COLORS, LEGEND

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').is_file())
output = root/'reports/hotspots_oct22'
observations = json.loads((output/'observations.json').read_text())
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
palette = np.zeros((256, 4), dtype='uint8')
for value, color in COLORS.items():
    palette[value] = color
for ax, record in zip(axes.flat, observations):
    raster = root/'data/scenes/hotspots_oct22'/record['site']/'reference/water_context.tif'
    with rasterio.open(raster) as src:
        ax.imshow(palette[src.read(1)], interpolation='nearest')
    ax.set_title(f"{record['site']} · {record['observed_utc'][8:10]} Oct", fontsize=11)
    ax.axis('off')
legend = [Patch(color=palette[k]/255, label=LEGEND[str(k)]) for k in COLORS]
fig.legend(handles=legend, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Water detected by radar across eight Nigerian hotspots', fontsize=16)
fig.subplots_adjust(top=0.90, bottom=0.13, hspace=0.16, wspace=0.08)
fig.savefig(output/'hotspot_observations.png', dpi=160, bbox_inches='tight')
plt.close(fig)